# Crop Disease Classifier — Google Colab Training Notebook

This notebook trains an **EfficientNet-B0** model on the **PlantVillage** dataset (38 classes) using a free **T4 GPU** on Google Colab.

### Steps:
1. Switch runtime to GPU (**Runtime → Change runtime type → T4 GPU**)
2. Download PlantVillage dataset
3. Perform stratified 70/15/15 train/val/test split
4. Train EfficientNet-B0 with custom 38-class head (Phase 1 warmup + Phase 2 fine-tuning)
5. Evaluate Macro F1 on held-out test split
6. Download `best_model.pth` and `classes.json` to your local project

In [ ]:
# 1. Verify GPU availability
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("Warning: Running on CPU! Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# 2. Clone PlantVillage dataset (raw color images, ~800MB)
!mkdir -p data/raw
!git clone --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git temp_pv
!cd temp_pv && git sparse-checkout set raw/color
!mv temp_pv/raw/color data/raw/plantvillage
!rm -rf temp_pv
!echo "Classes downloaded:" $(ls -1 data/raw/plantvillage | wc -l)
!echo "Total images:" $(find data/raw/plantvillage -type f | wc -l)

In [ ]:
# 3. Create Stratified Train / Val / Test Split (70/15/15)
import os, json
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

DATASET_DIR = Path("data/raw/plantvillage")
SPLITS_DIR = Path("data/splits")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

class_dirs = sorted([d for d in DATASET_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")])
class_to_idx = {cdir.name: idx for idx, cdir in enumerate(class_dirs)}
idx_to_class = {idx: cdir.name for idx, cdir in enumerate(class_dirs)}

with open(SPLITS_DIR / "classes.json", "w") as f:
    json.dump({"class_to_idx": class_to_idx, "idx_to_class": idx_to_class, "num_classes": len(class_dirs)}, f, indent=2)

records = []
fruit_crops = {"Apple", "Cherry_(including_sour)", "Grape", "Orange", "Peach", "Pepper,_bell", "Strawberry", "Tomato"}
for cdir in class_dirs:
    class_name = cdir.name
    class_idx = class_to_idx[class_name]
    crop = class_name.split("___")[0]
    cat = "Fruit" if crop in fruit_crops else "Plant"
    for img_p in cdir.iterdir():
        if img_p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            records.append({"image_path": str(img_p.resolve()), "class_name": class_name, "class_idx": class_idx, "crop": crop, "category": cat})

df = pd.DataFrame(records)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["class_idx"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["class_idx"], random_state=42)

train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)

print(f"Split complete! Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")

In [ ]:
# 4. PyTorch Dataset & Preprocessing
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PlantVillageDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["class_idx"])

train_loader = DataLoader(PlantVillageDataset(SPLITS_DIR / "train.csv", train_transform), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(PlantVillageDataset(SPLITS_DIR / "val.csv", eval_transform), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(PlantVillageDataset(SPLITS_DIR / "test.csv", eval_transform), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders initialized.")

In [ ]:
# 5. Build Model & Train
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(in_features, 38))
model = model.to(device)

# Freeze backbone for warmup
for p in model.features.parameters():
    p.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

best_macro_f1 = 0.0
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(1, 3):  # Warmup epochs
    model.train()
    tr_loss, correct, total = 0, 0, 0
    for imgs, lbls in tqdm(train_loader, desc=f"Warmup Epoch {epoch}"):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * imgs.size(0)
        correct += (outs.argmax(1) == lbls).sum().item()
        total += lbls.size(0)

    # Validate
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            all_p.extend(model(imgs).argmax(1).cpu().numpy())
            all_t.extend(lbls.cpu().numpy())
    val_f1 = f1_score(all_t, all_p, average="macro")
    print(f"Epoch {epoch} | Train Acc: {correct/total*100:.2f}% | Val Macro F1: {val_f1:.4f}")
    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        torch.save({"model_state_dict": model.state_dict(), "macro_f1": best_macro_f1}, "checkpoints/best_model.pth")
        print("  -> Saved best model!")

print("Training complete! Best Macro F1:", best_macro_f1)

In [ ]:
# 6. Evaluate on Test Set & Download Best Model
from google.colab import files

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(device)
        all_preds.extend(model(imgs).argmax(1).cpu().numpy())
        all_targets.extend(lbls.numpy())

test_f1 = f1_score(all_targets, all_preds, average="macro")
print(f"FINAL TEST MACRO F1: {test_f1:.4f}")

# Download weights and class mapping to your laptop
files.download("checkpoints/best_model.pth")
files.download("data/splits/classes.json")